# 02 — Building an LSTM Classifier

Train an LSTM to classify next-day direction from rolling windows.

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

torch.manual_seed(12)

In [ ]:
n, t, f = 900, 30, 8
X = torch.randn(n, t, f)
signal = X[:, :, 0].mean(dim=1) + 0.25 * X[:, :, 1].mean(dim=1)
y = (signal > 0).long()

dataset = TensorDataset(X, y)
train_ds, val_ds = random_split(dataset, [700, 200])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, num_layers=1, n_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(last)

model = LSTMClassifier(input_dim=f)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
for epoch in range(1, 16):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            preds = model(X[val_ds.indices]).argmax(1)
            acc = (preds == y[val_ds.indices]).float().mean().item()
        print(f'Epoch {epoch:>2} | val_acc={acc:.3f}')

## Exercises
1. Increase hidden size to 64 and compare validation accuracy.
2. Try 2 LSTM layers and observe runtime vs performance.